In [3]:
%matplotlib inline

In [5]:
pwd

'C:\\Users\\$8CUJ00-BGKRQC8B3RAL\\Desktop\\Plant Lab'

In [15]:
#Load datasets 
data_dir = 'C://Users//$8CUJ00-BGKRQC8B3RAL//Desktop//Plant Lab'

In [17]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Define the paths
data_dir = r'C://Users//$8CUJ00-BGKRQC8B3RAL//Desktop//Plant Lab//NonsegmentedV2'
output_dir = r'C://Users//$8CUJ00-BGKRQC8B3RAL//Desktop//Plant Lab//Split_Dataset'

# Create train and val directories
train_dir = os.path.join(output_dir, 'train')
val_dir = os.path.join(output_dir, 'val')

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# Iterate through each category
for category in os.listdir(data_dir):
    category_path = os.path.join(data_dir, category)
    if os.path.isdir(category_path):
        # Create category directories in train and val folders
        train_category_path = os.path.join(train_dir, category)
        val_category_path = os.path.join(val_dir, category)
        os.makedirs(train_category_path, exist_ok=True)
        os.makedirs(val_category_path, exist_ok=True)
        
        # Get all images in the category
        images = [img for img in os.listdir(category_path) if os.path.isfile(os.path.join(category_path, img))]
        
        # Split into train and val sets (80-20 split)
        train_images, val_images = train_test_split(images, test_size=0.2, random_state=42)
        
        # Move images to respective folders
        for img in train_images:
            shutil.copy(os.path.join(category_path, img), os.path.join(train_category_path, img))
        
        for img in val_images:
            shutil.copy(os.path.join(category_path, img), os.path.join(val_category_path, img))

print("Dataset split into train and val folders successfully!")


Dataset split into train and val folders successfully!


In [23]:
# Import necessary libraries
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data transformations
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),  # Resize to 224x224
        transforms.RandomHorizontalFlip(),  # Apply random horizontal flip
        transforms.ToTensor(),  # Convert image to Tensor
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Normalize
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Load datasets
data_dir = r'C://Users//$8CUJ00-BGKRQC8B3RAL//Desktop//Plant Lab//Split_Dataset'

# Use ImageFolder to load the data
image_datasets = {
    x: datasets.ImageFolder(root=f"{data_dir}/{x}", transform=data_transforms[x])
    for x in ['train', 'val']
}

# Create DataLoaders for the datasets
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=32, shuffle=True, num_workers=2),
    'val': DataLoader(image_datasets['val'], batch_size=32, shuffle=False, num_workers=2)
}

# Get dataset sizes
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
# Get class names
class_names = image_datasets['train'].classes

# Print dataset information
print("Dataset sizes:", dataset_sizes)
print("Class names:", class_names)


Dataset sizes: {'train': 4426, 'val': 1113}
Class names: ['Black-grass', 'Charlock', 'Cleavers', 'Common Chickweed', 'Common wheat', 'Fat Hen', 'Loose Silky-bent', 'Maize', 'Scentless Mayweed', 'Shepherd’s Purse', 'Small-flowered Cranesbill', 'Sugar beet']


In [25]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

# Data transformations
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Load datasets
data_dir = r'C://Users//$8CUJ00-BGKRQC8B3RAL//Desktop//Plant Lab//Split_Dataset'

image_datasets = {x: datasets.ImageFolder(root=f"{data_dir}/{x}", transform=data_transforms[x])
                  for x in ['train', 'val']}
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=16, shuffle=True, num_workers=2),
    'val': DataLoader(image_datasets['val'], batch_size=16, shuffle=False, num_workers=2)
}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes
num_classes = len(class_names)

# Model setup
model = models.resnet50(pretrained=True) 
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, num_classes)  # Adjusted to match dataset classes
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scaler = GradScaler()

# Training function
def train_model(model, criterion, optimizer, dataloaders, dataset_sizes, num_epochs=25):
    best_model_wts = model.state_dict()
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print('-' * 10)

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)

                optimizer.zero_grad()

                with autocast(), torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()

    print(f"Best val Acc: {best_acc:.4f}")
    model.load_state_dict(best_model_wts)
    return model

# Train the model
model = train_model(model, criterion, optimizer, dataloaders, dataset_sizes)

# Save the trained model
torch.save(model.state_dict(), "resnet50_multi_class_classification.pth")


C:\Users\$8CUJ00-BGKRQC8B3RAL\Desktop\Plant Lab\PlantLab\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\$8CUJ00-BGKRQC8B3RAL\Desktop\Plant Lab\PlantLab\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\$8CUJ00-BGKRQC8B3RAL\AppData\Local\Temp\ipykernel_18524\349368898.py:48: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
C:\Users\$8CUJ00-BGKRQC8B3RAL\Desktop\Plant Lab\PlantLab\Lib\site-packages

Epoch 1/25
----------


C:\Users\$8CUJ00-BGKRQC8B3RAL\AppData\Local\Temp\ipykernel_18524\349368898.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(), torch.set_grad_enabled(phase == 'train'):
C:\Users\$8CUJ00-BGKRQC8B3RAL\Desktop\Plant Lab\PlantLab\Lib\site-packages\torch\amp\autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


train Loss: 1.3716 Acc: 0.5436
val Loss: 1.0630 Acc: 0.6550
Epoch 2/25
----------
train Loss: 0.8085 Acc: 0.7253
val Loss: 1.2546 Acc: 0.6074
Epoch 3/25
----------
train Loss: 0.5927 Acc: 0.7928
val Loss: 0.9239 Acc: 0.7026
Epoch 4/25
----------
train Loss: 0.4661 Acc: 0.8337
val Loss: 0.4555 Acc: 0.8383
Epoch 5/25
----------
train Loss: 0.4489 Acc: 0.8437
val Loss: 0.5085 Acc: 0.8194
Epoch 6/25
----------
train Loss: 0.3689 Acc: 0.8751
val Loss: 0.4380 Acc: 0.8580
Epoch 7/25
----------
train Loss: 0.3410 Acc: 0.8769
val Loss: 0.3840 Acc: 0.8724
Epoch 8/25
----------
train Loss: 0.2866 Acc: 0.9006
val Loss: 0.4501 Acc: 0.8239
Epoch 9/25
----------
train Loss: 0.2836 Acc: 0.9040
val Loss: 0.3250 Acc: 0.8967
Epoch 10/25
----------
train Loss: 0.2338 Acc: 0.9130
val Loss: 0.3172 Acc: 0.8913
Epoch 11/25
----------
train Loss: 0.2187 Acc: 0.9189
val Loss: 0.3392 Acc: 0.8805
Epoch 12/25
----------
train Loss: 0.2023 Acc: 0.9254
val Loss: 0.6122 Acc: 0.8365
Epoch 13/25
----------
train Loss: 

In [45]:
from torchsummary import summary

# Print model summary
summary(model, input_size=(3, 224, 224))


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
              ReLU-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 64, 56, 56]           4,096
       BatchNorm2d-6           [-1, 64, 56, 56]             128
              ReLU-7           [-1, 64, 56, 56]               0
            Conv2d-8           [-1, 64, 56, 56]          36,864
       BatchNorm2d-9           [-1, 64, 56, 56]             128
             ReLU-10           [-1, 64, 56, 56]               0
           Conv2d-11          [-1, 256, 56, 56]          16,384
      BatchNorm2d-12          [-1, 256, 56, 56]             512
           Conv2d-13          [-1, 256, 56, 56]          16,384
      BatchNorm2d-14          [-1, 256,